In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
text = """
artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.

deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.

machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.

technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making.
"""

In [3]:
chars = sorted(list(set(text)))
char_to_idx = {ch:i for i,ch in enumerate(chars)}
idx_to_char = {i:ch for ch,i in char_to_idx.items()}

seq_length = 40
X = []
Y = []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[ch] for ch in text[i:i+seq_length]])
    Y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X)
Y = torch.tensor(Y)

In [4]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = LSTMModel(len(chars))

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(10):
    output = model(X)
    loss = criterion(output, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 3.337555408477783
Epoch 1, Loss: 3.2778048515319824
Epoch 2, Loss: 3.2153403759002686
Epoch 3, Loss: 3.142427444458008
Epoch 4, Loss: 3.0504586696624756
Epoch 5, Loss: 2.938192844390869
Epoch 6, Loss: 2.8740408420562744
Epoch 7, Loss: 2.8453118801116943
Epoch 8, Loss: 2.7832303047180176
Epoch 9, Loss: 2.7244999408721924


In [6]:
def generate_text(model, start_text, length=200):
    model.eval()
    result = start_text

    for _ in range(length):
        x = torch.tensor([[char_to_idx[ch] for ch in result[-seq_length:]]])
        output = model(x)
        pred = torch.argmax(output).item()
        result += idx_to_char[pred]

    return result

print(generate_text(model, "machine learning"))

machine learning atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin atin


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [8]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 64)
        self.pos = PositionalEncoding(64)

        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Linear(64, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.transformer(x)
        x = self.fc(x[:, -1, :])
        return x

model_t = TransformerModel(len(chars))

/tmp/ipykernel_3011/1704102061.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


In [9]:
optimizer = torch.optim.Adam(model_t.parameters(), lr=0.003)

for epoch in range(10):
    output = model_t(X)
    loss = criterion(output, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Transformer Epoch {epoch}, Loss: {loss.item()}")

Transformer Epoch 0, Loss: 3.495490074157715
Transformer Epoch 1, Loss: 3.0824718475341797
Transformer Epoch 2, Loss: 2.9606356620788574
Transformer Epoch 3, Loss: 2.935981273651123
Transformer Epoch 4, Loss: 2.8812549114227295
Transformer Epoch 5, Loss: 2.8052563667297363
Transformer Epoch 6, Loss: 2.736848831176758
Transformer Epoch 7, Loss: 2.659639835357666
Transformer Epoch 8, Loss: 2.597957134246826
Transformer Epoch 9, Loss: 2.544619083404541


In [10]:
print(generate_text(model_t, "artificial intelligence"))

artificial intelligence s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s
